# 🧬 AI-Driven Prediction of Mesenchymal Stem Cell Count
Machine Learning Pipeline Implementation

This program builds a predictive machine learning system to estimate Mesenchymal Stem Cell (MSC) proliferation based on culture conditions.

The pipeline includes:

📊 Dataset Processing

🧹 Data Preprocessing

🤖 Multiple ML Models

📈 Performance Evaluation

🧠 Neural Network Modeling


# 📦 1. Import Required Libraries

These libraries support data processing, machine learning models, and evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor

import xgboost as xgb

# 📊 2. Load the Dataset

The dataset contains 5000 MSC culture experiments with 19 biological parameters.

In [ ]:
data = pd.read_excel("msc_stem_cell_dataset_5000_samples.xlsx")

print("Dataset Shape:", data.shape)

Dataset Shape: (5000, 20)


# 🧬 3. Feature Selection (Biologically Important Parameters)

In stem cell culture systems, not all parameters contribute equally to cell proliferation. Certain biological factors directly influence growth kinetics, nutrient metabolism, and cell viability.

Therefore, a feature selection step is introduced to focus the model on high-impact biological variables.

This improves:

✨ Model accuracy

⚡ Training efficiency

🔬 Biological interpretability

# 🧪 Selected Biological Features

The following parameters are known to strongly affect MSC growth behavior:

| Feature              | Biological Role                          |
| -------------------- | ---------------------------------------- |
| **Seeding Density**  | Determines initial cell population       |
| **Confluence**       | Indicates cell coverage and growth stage |
| **Final Viability**  | Reflects health of cultured cells        |
| **Doubling Time**    | Measures cell proliferation rate         |
| **Glucose Level**    | Key energy source for cells              |
| **Growth Factor**    | Stimulates cell proliferation            |
| **Culture Duration** | Determines total growth period           |


In [ ]:
selected_features = [

"SeedingDensity_cells_cm2",
"Confluence_%",
"Final_Viability_%",
"Doubling_Time_hours",
"Glucose_g_L",
"GrowthFactor_ng_mL",
"Culture_Duration_days"

]

# Select only the biologically important features
X = data[selected_features]

# Target variable
y = data["Stem Cell Count"]

# 🔀 4. Train-Test Split

The dataset is divided into:

📚 Training Data (80%)

🧪 Testing Data (20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 🧹 5. Missing Value Handling (KNN Imputation)

The dataset includes 7% missing values to simulate industrial biological datasets.

We use KNN Imputer, which replaces missing values based on nearest biological samples.

In [ ]:
imputer = KNNImputer(n_neighbors=5)

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# ⚖️ 6. Feature Scaling

Biological parameters have different units:

°C

%

ng/mL

cells/cm²

Standardization ensures all features contribute equally.

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 🎯 7. Target Normalization

Normalizing the target improves training stability.

In [ ]:
target_scaler = StandardScaler()

y_train = target_scaler.fit_transform(
    y_train.values.reshape(-1,1)
).ravel()

y_test = target_scaler.transform(
    y_test.values.reshape(-1,1)
).ravel()

# 🤖 8. Machine Learning Models

Four models are evaluated.



# 🌳 Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=1000,
    max_depth=25,
    random_state=42
)

# 📈 Gradient Boosting Regressor

In [ ]:
gbr_model = GradientBoostingRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6
)

# 🚀 XGBoost Regressor

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

# 🧠 Neural Network (MLP)

Architecture

Input Layer → 128 → 64 → 32 → Output

In [ ]:
mlp_model = MLPRegressor(

    hidden_layer_sizes=(128,64,32),
    activation='relu',
    solver='adam',
    batch_size=32,
    learning_rate='adaptive',
    max_iter=500,
    random_state=42

)

# 🏋️ 9. Train Models

In [ ]:
print("\nTraining Models...\n")

rf_model.fit(X_train, y_train)
gbr_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)
mlp_model.fit(X_train, y_train)


Training Models...



MLPRegressor(batch_size=32, hidden_layer_sizes=(128, 64, 32),
             learning_rate='adaptive', max_iter=500, random_state=42)

# 🔮 10. Generate Predictions

In [ ]:
rf_pred = rf_model.predict(X_test)
gbr_pred = gbr_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)
mlp_pred = mlp_model.predict(X_test)

# 🤝 11. Ensemble Model

Combines predictions from multiple models.

In [ ]:
ensemble_pred = (
    0.35 * rf_pred +
    0.35 * xgb_pred +
    0.15 * gbr_pred +
    0.15 * mlp_pred
)

# 📏 12. Evaluation Function

In [ ]:
def evaluate_model(y_true, y_pred, name):

    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    print(name)
    print("R2 Score :", r2)
    print("RMSE :", rmse)
    print("-----------------------")

# 📊 13. Model Performance

In [ ]:
print("\nModel Performance\n")

evaluate_model(y_test, rf_pred, "Random Forest")
evaluate_model(y_test, gbr_pred, "Gradient Boosting")
evaluate_model(y_test, xgb_pred, "XGBoost")
evaluate_model(y_test, mlp_pred, "Neural Network (MLP)")
evaluate_model(y_test, ensemble_pred, "Final Ensemble Model")


Model Performance

Random Forest
R2 Score : 0.9898948559191609
RMSE : 0.10261153099005307
-----------------------
Gradient Boosting
R2 Score : 0.9920999552393124
RMSE : 0.09072765549253757
-----------------------
XGBoost
R2 Score : 0.9951010803323839
RMSE : 0.07144553149285159
-----------------------
Neural Network (MLP)
R2 Score : 0.9938463206105578
RMSE : 0.08007415144939871
-----------------------
Final Ensemble Model
R2 Score : 0.9947152227714751
RMSE : 0.07420586892486764
-----------------------


# 🔍 14. Cross-Validation

Ensures the model generalizes well.

In [ ]:
print("\nCross Validation (Random Forest)\n")

cv_scores = cross_val_score(
    rf_model,
    X_train,
    y_train,
    cv=10,
    scoring="r2"
)

print("Average CV R2:", cv_scores.mean())


Cross Validation (Random Forest)

Average CV R2: 0.9830237595363573


# 🔄 15. Convert Predictions to Original Scale

In [ ]:
ensemble_pred_original = target_scaler.inverse_transform(
    ensemble_pred.reshape(-1,1)
)

y_test_original = target_scaler.inverse_transform(
    y_test.reshape(-1,1)
)

# 🔎 16. Display Sample Predictions

In [ ]:
print("\nExample Predictions\n")

for i in range(5):

    print(
        "Actual:", y_test_original[i][0],
        "| Predicted:", ensemble_pred_original[i][0]
    )


Example Predictions

Actual: 13904.4 | Predicted: 13910.820019335722
Actual: 10668.5 | Predicted: 10754.379057172273
Actual: 13581.7 | Predicted: 13387.107984094411
Actual: 13838.9 | Predicted: 13853.010764928895
Actual: 11803.8 | Predicted: 11827.144115947716


# 🏆 Final Outcome

Your project successfully builds an AI-driven predictive model for stem cell proliferation.

Best model performance:

⭐ XGBoost Regressor

R² ≈ 0.995

RMSE ≈ 0.07

This means the model explains ~99.5% of variance in MSC count, demonstrating extremely strong predictive capability.

# 🌟 Potential Applications

🧬 Predict stem cell growth in culture

⚙️ Optimize bioreactor parameters

📊 AI-assisted regenerative medicine research

🔬 Intelligent laboratory decision support

# 🧪 Interactive Model Testing (User Input)

This section allows the user to manually input MSC culture conditions, and the trained models will predict the Stem Cell Count.

In [ ]:
# ============================================================
# User Input for Culture Parameters
# ============================================================

print("\nEnter Culture Conditions for Prediction\n")

SeedingDensity = float(input("Seeding Density (cells/cm²): "))
Confluence = float(input("Confluence (%): "))
FinalViability = float(input("Final Viability (%): "))
DoublingTime = float(input("Doubling Time (hours): "))
Glucose = float(input("Glucose concentration (g/L): "))
GrowthFactor = float(input("Growth Factor (ng/mL): "))
CultureDuration = float(input("Culture Duration (days): "))

# ============================================================
# Convert Inputs to DataFrame
# ============================================================

test_sample = {

"SeedingDensity_cells_cm2": SeedingDensity,
"Confluence_%": Confluence,
"Final_Viability_%": FinalViability,
"Doubling_Time_hours": DoublingTime,
"Glucose_g_L": Glucose,
"GrowthFactor_ng_mL": GrowthFactor,
"Culture_Duration_days": CultureDuration

}

test_df = pd.DataFrame([test_sample])

# ============================================================
# Apply Scaling
# ============================================================

test_scaled = scaler.transform(test_df)

# ============================================================
# Model Predictions
# ============================================================

rf_prediction = rf_model.predict(test_scaled)
gbr_prediction = gbr_model.predict(test_scaled)
xgb_prediction = xgb_model.predict(test_scaled)
mlp_prediction = mlp_model.predict(test_scaled)

rf_cells = target_scaler.inverse_transform(rf_prediction.reshape(-1,1))[0][0]
gbr_cells = target_scaler.inverse_transform(gbr_prediction.reshape(-1,1))[0][0]
xgb_cells = target_scaler.inverse_transform(xgb_prediction.reshape(-1,1))[0][0]
mlp_cells = target_scaler.inverse_transform(mlp_prediction.reshape(-1,1))[0][0]

ensemble_prediction = (
0.35 * rf_prediction +
0.35 * xgb_prediction +
0.15 * gbr_prediction +
0.15 * mlp_prediction
)

ensemble_cells = target_scaler.inverse_transform(
ensemble_prediction.reshape(-1,1)
)[0][0]

print("\n🔬 Predicted Stem Cell Count\n")

print("🌳 Random Forest:", rf_cells)
print("📈 Gradient Boosting:", gbr_cells)
print("🚀 XGBoost:", xgb_cells)
print("🧠 Neural Network:", mlp_cells)

print("\n🏆 Final Ensemble Prediction:", ensemble_cells)


Enter Culture Conditions for Prediction

Seeding Density (cells/cm²): 9000
Confluence (%): 95
Final Viability (%): 97
Doubling Time (hours): 16
Glucose concentration (g/L): 5.5
Growth Factor (ng/mL): 20
Culture Duration (days): 6

🔬 Predicted Stem Cell Count

🌳 Random Forest: 14502.396299999995
📈 Gradient Boosting: 14944.931347030575
🚀 XGBoost: 14433.327
🧠 Neural Network: 15322.216449841446

🏆 Final Ensemble Prediction: 14667.575277485446


# 🧬 Reverse Prediction: Target Cell Count Optimization

# ⌨️ Step 1 — User Inputs Target Cell Count

In [35]:
# ============================================================
# User Input: Desired Stem Cell Count
# ============================================================

print("\n🧬 Reverse Prediction: Culture Optimization\n")

target_cells = float(input("Enter desired stem cell count: "))

print("\nTarget Cell Count:", target_cells)


🧬 Reverse Prediction: Culture Optimization

Enter desired stem cell count: 1000000000

Target Cell Count: 1000000000.0


# 🔬 Step 2 — Define Biological Parameter Ranges

These ranges represent realistic MSC culture conditions.

In [36]:
ranges = {

"SeedingDensity_cells_cm2": (3000,10000),
"Confluence_%": (70,100),
"Final_Viability_%": (85,100),
"Doubling_Time_hours": (12,30),
"Glucose_g_L": (4,6),
"GrowthFactor_ng_mL": (10,30),
"Culture_Duration_days": (3,8)

}

# ⚙️ Step 3 — Reverse Optimization Search

The algorithm explores thousands of possible culture combinations.

In [ ]:
import numpy as np

best_result = None
best_prediction = None
best_error = float("inf")

for i in range(200000):

    sample = {

    "SeedingDensity_cells_cm2": np.random.uniform(3000,10000),
    "Confluence_%": np.random.uniform(70,100),
    "Final_Viability_%": np.random.uniform(85,100),
    "Doubling_Time_hours": np.random.uniform(12,30),
    "Glucose_g_L": np.random.uniform(4,6),
    "GrowthFactor_ng_mL": np.random.uniform(10,30),
    "Culture_Duration_days": np.random.uniform(3,8)

    }

    sample_df = pd.DataFrame([sample])

    sample_scaled = scaler.transform(sample_df)

    prediction = xgb_model.predict(sample_scaled)

    predicted_cells = target_scaler.inverse_transform(
        prediction.reshape(-1,1)
    )[0][0]

    error = abs(predicted_cells - target_cells)

    if error < best_error:

        best_error = error
        best_result = sample
        best_prediction = predicted_cells

# 📊 Step 4 — Display Optimal Culture Conditions

In [ ]:
# ============================================================
# Display Optimized Parameters
# ============================================================

print("\n🔬 Recommended Culture Conditions\n")

for key,value in best_result.items():

    print(key,":", round(value,2))

print("\n🧬 Predicted Cell Yield:", best_prediction)
print("🎯 Target Cell Count:", target_cells)